In [ ]:
### string complexity

def get_complexity(df):
    df['model_complexity'] = df['model'].str.len()
    df['engine_complexity'] = df['engine'].str.len()
    df['ext_col_complexity'] = df['ext_col'].str.len()
    df['int_col_complexity'] = df['int_col'].str.len()
    df['transmission_complexity'] = df['transmission'].str.len()
    return df

XY= get_complexity(XY)

In [ ]:
### regex pattern recognition

def get_expert_features(df):
    ### using regex patterns
    pattern = r'(150|250|350|450|550|650)'
    df['model_pickup'] = df['model'].str.extract(pattern).fillna(0).astype(int).astype('category')

    pattern = r'(mpfi|gtdi|tfsi|gdi|pdi|ddi|sidi|di)'
    df['fuel_inject'] = df['engine'].str.extract(pattern).fillna('asp').astype('category')

    pattern = r'(\d+(?:\.\d+)?)hp'
    df['horsepower'] = df['engine'].str.extract(pattern).fillna(-1).astype(float)

    pattern = r'(\d+(?:\.\d+)?)(?=l|\s*lit)'
    df['disp'] = df['engine'].str.extract(pattern).fillna(-1).astype(float)

    pattern = r'([vihsfw]\d+|\d+\s*cyl|straight\s*\d+|flat\s*\d+)'
    df['engine_type'] = df['engine'].str.extract(pattern).fillna('unk').astype('category')
    pattern = r'(\d+)'
    df['cylinders'] = df['engine_type'].str.extract(pattern).fillna(-1).astype(int)

    pattern = r'(\d+(?:\.\d+)?)(?=v)'
    df['valves'] = df['engine'].str.extract(pattern).fillna(-1).astype(int).astype('category')
    df['valves'].replace({697: -1}, inplace=True)

    pattern = r'(turbo|super|cool)'
    df['turbo'] = df['engine'].str.extract(pattern).notna()

    pattern = r'(dohc|sohc|ohv)'
    df['cams'] = df['engine'].str.extract(pattern).fillna('-1').astype('category')

    pattern = r'(\d+)'
    df['gears'] = df['transmission'].str.extract(pattern).fillna(-1).astype(int)

    pattern = r'(cvt|man|auto)'
    df['trans'] = df['transmission'].str.extract(pattern).fillna('-1').astype('category')

    pattern = r'(metallic|coat|cool)'
    df['special_paint'] = df['ext_col'].str.extract(pattern).notna()

    ### consider implied fuel - using engine to infer fuel when fuel is not explicit
    df['age'] = 2026 - df['model_year']
    df['milesperyear'] =  df['milage'] / df['age']
    
    return df

XY= get_expert_features(XY)

In [ ]:
#### Generate Expert features based on research and pca loadings

def get_domain_expert_features(df):
    ### healthy height weight ratio -> PCA 7
    df['bmi'] = df['weight']/ ((df['height']/100) **2) 
    df['height_weight_ratio'] = df['weight']/df['height']
    ###https://www.calculator.net/ideal-weight-calculator.html
    # Male:   48.0 kg + 2.7 kg per inch over 5 feet
    # Female: 45.5 kg + 2.2 kg per inch over 5 feet
    df['male'] = df['sex'].replace({1:1, -1:0})
    df['hanwi'] = df['weight'] - (45.5 + (df['male'] * (48-45.5)) + (2.2 + (df['sex'] * (2.7-2.2)))*(df['height']-152.4)/2.54)
    # Male:	  56.2 kg + 1.41 kg per inch over 5 feet
    # Female: 53.1 kg + 1.36 kg per inch over 5 feet
    df['miller'] = df['weight'] - (53.1 + (df['male'] * (56.2-53.1)) + (1.36 + (df['sex'] * (1.41-1.36)))*(df['height']-152.4))
    df.drop('male', inplace=True, axis=1)
    feats = ['bmi', 'height_weight_ratio', 'hanwi', 'miller']
    df = mkf.get_transformed_features(df, feats, skl.preprocessing.StandardScaler(), winsorize=[0.001, 0.001])
    ### temp vs heart rate -> PCA 5
    df['vital_ratio'] = df['heart_rate'] / df['body_temp']
    ## delta from reference
    df['body_temp'] +=  -37
    df['heart_rate'] +=  -60
    ### effort 
    df['effort'] = df['heart_rate'] * df['body_temp'] * df['duration']
    ### effort per size -> PCA 1
    df['size_effort'] = df['effort'] / (df['weight'] * df['height'])
    ### total effort -> PCA 2
    df['max_effort'] = df['effort'] * df['weight'] * df['height']
    ### workout effort per unit time -> PCA 6
    df['burn_rate'] = df['body_temp'] * df['heart_rate'] / df['duration'] 
    feats = ['effort', 'size_effort', 'max_effort', 'vital_ratio', 'burn_rate']
    df = mkf.get_transformed_features(df, feats , skl.preprocessing.PowerTransformer(), winsorize=[0.001, 0.001])
    XY['temp_2'] = XY['body_temp'] ** 2
    XY['temp_3'] = XY['body_temp'] ** 3
    XY['temp_4'] = XY['body_temp'] ** 4
    feats = ['temp_2', 'temp_3', 'temp_4']
    df = mkf.get_transformed_features(df, feats , skl.preprocessing.MinMaxScaler())
    return df

XY = get_domain_expert_features(XY)

In [ ]:
### lag features for time series data
### need to clean and validate

### get two days of history and the difference (change per day) in the feature
def get_history(df, feature, delta_only = True):
    #yesterday's feature
    df[f'{feature}_yesterday'] = df[feature].shift(periods = 1)
    df[f'{feature}_yesterday'].fillna(method='bfill', inplace = True)
    #feature 2 days prior
    df[f'{feature}_tdby'] = df[feature].shift(periods = 2)
    df[f'{feature}_tdby'].fillna(method='bfill', inplace = True)
    #change from yesterday and change from 2 days ago
    df[f'{feature}_yesterday_delta'] = df[feature] - df[f'{feature}_yesterday']
    df[f'{feature}_tdby_delta'] = 0.5 * (df[feature] - df[f'{feature}_tdby'])
    if delta_only == True:
        df.drop(f'{feature}_yesterday', inplace = True, axis = 1)
        df.drop(f'{feature}_tdby', inplace = True, axis = 1)
    return df